# Natural Language Processing - Text Preprocessing

## Libraries and settings

In [1]:
# Libraries
import os
import re
import string
import numpy as np
import pandas as pd
from pprint import pprint

import nltk

# Import only once
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

from nltk.tag import pos_tag
from nltk.corpus import stopwords
from nltk.chunk import tree2conlltags
from nltk.chunk import conlltags2tree
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Current working directory
print('Current working directory:', os.getcwd())

[nltk_data] Downloading package stopwords to /home/vscode/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /home/vscode/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /home/vscode/nltk_data...
[nltk_data] Downloading package omw-1.4 to /home/vscode/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/vscode/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


Current working directory: /workspaces/data_analytics/Week_11


## Defining documents

In [17]:
# Defining documents (=sentenses)
d1 = "Data analytics is fascinating,fun but complex."
d2 = "I love learning Python and machine learning."
d3 = "Natural language processing is a very cool topic."

corpus_01 = d1 + ' ' + d2 + ' ' + d3
corpus_01

'Data analytics is fascinating,fun but complex. I love learning Python and machine learning. Natural language processing is a very cool topic.'

## Text preprocessing
#### Steps:
- Text to lowercase
- Removing punctuations
- Tokenization
- Removal of stop words
- Lemmatization

### Text to lowercase

In [18]:
# Text to lowercase function
def text_lowercase(text):
    return text.lower()

# Text to lowercase
corpus_02 = text_lowercase(corpus_01)
corpus_02

'data analytics is fascinating,fun but complex. i love learning python and machine learning. natural language processing is a very cool topic.'

### Removing punctuation

In [19]:
# Remove punctuation function
def remove_punctuation(text):
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

# Remove punctuation
corpus_03 = remove_punctuation(corpus_02)
corpus_03

'data analytics is fascinatingfun but complex i love learning python and machine learning natural language processing is a very cool topic'

### Tokenize text & removal of stopwords

In [20]:
# Show english stopwords
eng_stopwords = set(stopwords.words('english'))
print("List of english stopwords:")
print(eng_stopwords)

List of english stopwords:
{"he'd", 'more', "shouldn't", 'then', 'won', 'ma', 'nor', "it's", 'hasn', 'ours', "isn't", 'yours', 'me', 'is', "he'll", "doesn't", 'you', "it'd", 'this', 'should', 'or', 'myself', 'they', 'whom', 'll', "i'd", "they're", 'does', 'wouldn', 'be', 'at', 'in', 'd', 'mustn', 'having', 'only', "you've", 'did', 'he', 'we', 'where', 'of', 'those', 'him', 'about', "mustn't", "won't", "you'll", 'himself', "we'd", 'below', 'haven', 'its', 'few', 's', 'now', 'than', 'until', "weren't", 'just', "that'll", 'while', 'all', 'itself', 'ain', 'been', 'a', "didn't", 'during', 'off', 'with', 'if', 'so', 'which', 'his', 'both', 'do', 'don', 'out', 'she', 'under', "i'll", "aren't", "i'm", 'same', 'but', 'theirs', 'herself', "you'd", 'your', 'further', 'will', 'to', 'an', "shan't", 'above', 'has', 'over', 'are', 'each', 'ourselves', 'who', 'the', 'shouldn', 'o', 're', "don't", 'i', 'being', "wasn't", 'very', 'such', 'have', 'any', 'before', 'were', "she's", 'them', "mightn't", "sho

In [21]:
# Function for tokenization and the removal of stopwords
def remove_stopwords(text):
    stop_words = set(stopwords.words("english"))
    word_tokens = word_tokenize(text)
    filtered_text = [word for word in word_tokens if word not in stop_words]
    return filtered_text
 
# Remove stopwords
corpus_04 = remove_stopwords(corpus_03)
print(corpus_04, end="")

['data', 'analytics', 'fascinatingfun', 'complex', 'love', 'learning', 'python', 'machine', 'learning', 'natural', 'language', 'processing', 'cool', 'topic']

### Lemmatization

In [24]:
# Initialize Lemmatizer
lemmatizer = WordNetLemmatizer()

# Lemmatize string function
def lemmatize_word(text):
    word_tokens = word_tokenize(text)
    lemmas = [lemmatizer.lemmatize(word, pos ='v') for word in word_tokens]
    return lemmas

# Lemmatize
lem = []
for i in corpus_04:
    lem.append(lemmatize_word(i))

# Nested list to list
corpus_05 = [' '.join([str(x) for x in lst]) for lst in lem]

print('Before lemmatization:')
print(corpus_04, '\n')

print('After lemmatization:')
print(corpus_05, end="")

Before lemmatization:
['data', 'analytics', 'fascinatingfun', 'complex', 'love', 'learning', 'python', 'machine', 'learning', 'natural', 'language', 'processing', 'cool', 'topic'] 

After lemmatization:
['data', 'analytics', 'fascinatingfun', 'complex', 'love', 'learn', 'python', 'machine', 'learn', 'natural', 'language', 'process', 'cool', 'topic']

## Redefine the text corpus (pre-processed)

In [25]:
# We will use the lemmatized words above to re-define our corpus 
corpus = ['data analytics fascinatefun complex', 
          'love learn python machine learn', 
          'natural language process cool topic']

## Document-term matrix with ngram_range=(1,1)

In [26]:
# Vectorizer with ngram_range=(1,1)
vectorizer = CountVectorizer(min_df=0.0, ngram_range=(1,1))

# Transform 
count = vectorizer.fit_transform(corpus)
 
# Create dataframe
df_count = pd.DataFrame(count.toarray(),
                        columns=vectorizer.get_feature_names_out())

print('Document-term matrix')
print(df_count)

Document-term matrix
   analytics  complex  cool  data  fascinatefun  language  learn  love  \
0          1        1     0     1             1         0      0     0   
1          0        0     0     0             0         0      2     1   
2          0        0     1     0             0         1      0     0   

   machine  natural  process  python  topic  
0        0        0        0       0      0  
1        1        0        0       1      0  
2        0        1        1       0      1  


## Document-term matrix with ngram_range=(2,2)

In [27]:
# Vectorizer with with ngram_range=(2,2)
vectorizer = CountVectorizer(min_df=0.0, ngram_range=(2,2))

# Transform 
count = vectorizer.fit_transform(corpus)
 
# Create dataframe
df_count = pd.DataFrame(count.toarray(),
                        columns=vectorizer.get_feature_names_out())

print('Document-term matrix')
print(df_count)

Document-term matrix
   analytics fascinatefun  cool topic  data analytics  fascinatefun complex  \
0                       1           0               1                     1   
1                       0           0               0                     0   
2                       0           1               0                     0   

   language process  learn python  love learn  machine learn  \
0                 0             0           0              0   
1                 0             1           1              1   
2                 1             0           0              0   

   natural language  process cool  python machine  
0                 0             0               0  
1                 0             0               1  
2                 1             1               0  


## Term frequency-inverse document frequency (TF-IDF)
- For details see: https://www.learndatasci.com/glossary/tf-idf-term-frequency-inverse-document-frequency

### Term Frequency (TF)

In [28]:
# Compute Term Frequency (TF)
words_set = set()
for doc in corpus:
    words = doc.split(' ')
    words_set = words_set.union(set(words))
    
print('Number of words in the corpus:',len(words_set), '\n')
print('The words in the corpus: \n', words_set)

# Number of documents in the corpus
n_docs = len(corpus)

# Number of unique words in the corpus 
n_words_set = len(words_set)

df_tf = pd.DataFrame(np.zeros((n_docs, n_words_set)), 
                     columns=list(words_set))

print("\nTerm Frequency (TF):")
for i in range(n_docs):
    # Words in the document
    words = corpus[i].split(' ')
    for w in words:
        df_tf[w][i] = df_tf[w][i] + (1 / len(words))
        
print(df_tf.round(4))

Number of words in the corpus: 13 

The words in the corpus: 
 {'cool', 'love', 'learn', 'topic', 'process', 'python', 'analytics', 'language', 'fascinatefun', 'complex', 'natural', 'machine', 'data'}

Term Frequency (TF):
   cool  love  learn  topic  process  python  analytics  language  \
0   0.0   0.0    0.0    0.0      0.0     0.0       0.25       0.0   
1   0.0   0.2    0.4    0.0      0.0     0.2       0.00       0.0   
2   0.2   0.0    0.0    0.2      0.2     0.0       0.00       0.2   

   fascinatefun  complex  natural  machine  data  
0          0.25     0.25      0.0      0.0  0.25  
1          0.00     0.00      0.0      0.2  0.00  
2          0.00     0.00      0.2      0.0  0.00  


### Inverse Document Frequency (IDF)

In [29]:
# Computing Inverse Document Frequency (IDF)
print("\nInverse Document Frequency (IDF):")

idf = {}

for w in words_set:
    
    # k = number of documents that contain this word
    k = 0
    
    for i in range(n_docs):
        if w in corpus[i].split():
            k += 1
            
    idf[w] =  np.log10(n_docs / k).round(4)
    
    print(f'{w:>15}: {idf[w]:>10}')


Inverse Document Frequency (IDF):
           cool:     0.4771
           love:     0.4771
          learn:     0.4771
          topic:     0.4771
        process:     0.4771
         python:     0.4771
      analytics:     0.4771
       language:     0.4771
   fascinatefun:     0.4771
        complex:     0.4771
        natural:     0.4771
        machine:     0.4771
           data:     0.4771


### Term Frequency - Inverse Document Frequency (TF-IDF)

In [30]:
# Computing TF-IDF
df_tf_idf = df_tf.copy()

for w in words_set:
    for i in range(n_docs):
        df_tf_idf[w][i] = df_tf[w][i] * idf[w]

print('\nTF-IDF:')
print(df_tf_idf.round(4))


TF-IDF:
     cool    love   learn   topic  process  python  analytics  language  \
0  0.0000  0.0000  0.0000  0.0000   0.0000  0.0000     0.1193    0.0000   
1  0.0000  0.0954  0.1908  0.0000   0.0000  0.0954     0.0000    0.0000   
2  0.0954  0.0000  0.0000  0.0954   0.0954  0.0000     0.0000    0.0954   

   fascinatefun  complex  natural  machine    data  
0        0.1193   0.1193   0.0000   0.0000  0.1193  
1        0.0000   0.0000   0.0000   0.0954  0.0000  
2        0.0000   0.0000   0.0954   0.0000  0.0000  


## Part-of-Speach (POS) tagging
For meaning of POS-tags see: https://pythonexamples.org/nltk-pos-tagging

In [31]:
text = '''Data scientists analyse huge datasets using Python more effectively. 
but complex problems require advanced techniques. Sometimes, they also need to visualize results.
depends how data scientists preprocess'''

def preprocess(sent):
    sent = nltk.word_tokenize(sent)
    sent = nltk.pos_tag(sent)
    return sent

sent = preprocess(text)
pattern = 'NP: {<DT>?<JJ>*<NN>}'

cp = nltk.RegexpParser(pattern)
cs = cp.parse(sent)

iob_tagged = tree2conlltags(cs)

# Print the POS-tags
pprint(iob_tagged)

[('Data', 'NNP', 'O'),
 ('scientists', 'NNS', 'O'),
 ('analyse', 'VBP', 'O'),
 ('huge', 'JJ', 'O'),
 ('datasets', 'NNS', 'O'),
 ('using', 'VBG', 'O'),
 ('Python', 'NNP', 'O'),
 ('more', 'RBR', 'O'),
 ('effectively', 'RB', 'O'),
 ('.', '.', 'O'),
 ('but', 'CC', 'O'),
 ('complex', 'JJ', 'O'),
 ('problems', 'NNS', 'O'),
 ('require', 'VBP', 'O'),
 ('advanced', 'JJ', 'O'),
 ('techniques', 'NNS', 'O'),
 ('.', '.', 'O'),
 ('Sometimes', 'RB', 'O'),
 (',', ',', 'O'),
 ('they', 'PRP', 'O'),
 ('also', 'RB', 'O'),
 ('need', 'VBP', 'O'),
 ('to', 'TO', 'O'),
 ('visualize', 'VB', 'O'),
 ('results', 'NNS', 'O'),
 ('.', '.', 'O'),
 ('depends', 'VBZ', 'O'),
 ('how', 'WRB', 'O'),
 ('data', 'JJ', 'O'),
 ('scientists', 'NNS', 'O'),
 ('preprocess', 'NN', 'B-NP')]


### Explanation

* **NNS (Noun, plural):** Refers to multiple objects or people.
    * *Examples from text:* "scientists", "datasets", "problems", "techniques", "results".
* **JJ (Adjective):** Describes or modifies a noun.
    * *Examples from text:* "huge", "complex", "advanced".
* **VBP (Verb, non-3rd person singular present):** An action verb in the present tense (matching a plural subject).
    * *Examples from text:* "analyse", "require".
* **NNP (Proper noun, singular):** A specific name of a person, place, or organization.
    * *Example from text:* "Python".
* **RB (Adverb):** Modifies a verb, adjective, or other adverb (describing how or when).
    * *Examples from text:* "effectively", "Sometimes", "also".
* **VBG (Verb, gerund/present participle):** A verb ending in -ing used as a modifier or continuous action.
    * *Example from text:* "using".

### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [15]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1030-azure
Datetime: 2025-12-13 10:02:54
Python Version: 3.11.13
-----------------------------------
